In [ ]:
import glob
import rawpy
import matplotlib.pyplot as plt

raw_dir = "/mnt/c/Users/crueh/Pictures/Raw/2026_08_12"
raw_files = sorted(glob.glob(f"{raw_dir}/*.CR3"))

first_raw = raw_files[0]
print(first_raw)

with rawpy.imread(first_raw) as raw:
    rgb = raw.postprocess()

plt.figure(figsize=(10, 8))
plt.imshow(rgb)
plt.axis("off")
plt.show()

In [ ]:
import base64
from io import BytesIO
from PIL import Image
from IPython.display import HTML

thumbs = []
for i, path in enumerate(raw_files):
    with rawpy.imread(path) as raw:
        img = raw.postprocess(half_size=True)
    im = Image.fromarray(img)
    im.thumbnail((480, 480))
    buf = BytesIO()
    im.save(buf, format="JPEG", quality=70)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    thumbs.append(f"data:image/jpeg;base64,{b64}")
    print(f"\r{i + 1}/{len(raw_files)}", end="")

frames_js = ",".join(f'"{t}"' for t in thumbs)
names_js = ",".join(f'"{path.split("/")[-1]}"' for path in raw_files)

html = f"""
<div>
  <img id="slideshow-img" src="{thumbs[0]}" style="max-width:480px; display:block;">
  <input id="slideshow-range" type="range" min="0" max="{len(thumbs) - 1}" value="0" style="width:480px;">
  <div id="slideshow-label"></div>
</div>
<script>
(function() {{
  const frames = [{frames_js}];
  const names = [{names_js}];
  const img = document.getElementById("slideshow-img");
  const range = document.getElementById("slideshow-range");
  const label = document.getElementById("slideshow-label");
  function update(i) {{
    img.src = frames[i];
    label.textContent = (parseInt(i) + 1) + " / " + frames.length + " — " + names[i];
  }}
  range.addEventListener("input", (e) => update(e.target.value));
  update(0);
}})();
</script>
"""
HTML(html)

In [ ]:
import numpy as np
import cv2

def fit_circle(points):
    """Least-squares (Kasa) circle fit through 2D points -> (cx, cy, r)."""
    x = points[:, 0].astype(np.float64)
    y = points[:, 1].astype(np.float64)
    A = np.column_stack([2 * x, 2 * y, np.ones_like(x)])
    b = x**2 + y**2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy, c = sol
    r = np.sqrt(c + cx**2 + cy**2)
    return cx, cy, r

def find_sun_circle(gray):
    """Threshold the sun/crescent, take the convex hull of its contour
    (drops the concave bite where the moon overlaps) and fit a circle
    to the remaining points -- this stays anchored to the sun's true
    limb regardless of how much the moon covers."""
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(largest).reshape(-1, 2)
    return fit_circle(hull)

# Pass 1: decode each frame (half size, same as the slideshow) and locate the sun.
frames = []
circles = []
for i, path in enumerate(raw_files):
    with rawpy.imread(path) as raw:
        img = raw.postprocess(half_size=True)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    cx, cy, r = find_sun_circle(gray)
    frames.append(img)
    circles.append((cx, cy, r))
    print(f"\r{i + 1}/{len(raw_files)}", end="")

circles = np.array(circles)
target_cx, target_cy = np.median(circles[:, 0]), np.median(circles[:, 1])
median_r = np.median(circles[:, 2])
crop_r = int(median_r * 1.3)

print(f"\ntarget center: ({target_cx:.1f}, {target_cy:.1f}), median radius: {median_r:.1f}px")

# Pass 2: shift every frame so its sun center lands on the target, then crop.
aligned = []
for img, (cx, cy, r) in zip(frames, circles):
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

print(f"aligned {len(aligned)} frames, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
def get_hull_points(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    return cv2.convexHull(largest).reshape(-1, 2).astype(np.float64)

def fit_center_fixed_radius(points, r, cx0, cy0, iters=15):
    """Gauss-Newton refinement of the center only, radius held fixed.
    Much better conditioned than a free 3-parameter fit when the sun's
    limb is just a thin sliver (deep partial phases)."""
    cx, cy = cx0, cy0
    for _ in range(iters):
        dx = points[:, 0] - cx
        dy = points[:, 1] - cy
        d = np.sqrt(dx**2 + dy**2)
        d = np.where(d < 1e-6, 1e-6, d)
        e = d - r
        J = np.column_stack([-dx / d, -dy / d])
        try:
            delta = np.linalg.solve(J.T @ J, -(J.T @ e))
        except np.linalg.LinAlgError:
            break
        cx, cy = cx + delta[0], cy + delta[1]
        if np.linalg.norm(delta) < 1e-4:
            break
    return cx, cy

def rolling_median(x, window=5):
    half = window // 2
    return np.array([np.median(x[max(0, i - half):i + half + 1]) for i in range(len(x))])

# Radius from the best-conditioned frames only (widest visible arc = most hull points)
hull_points_list = []
hull_counts = []
for img in frames:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    pts = get_hull_points(gray)
    hull_points_list.append(pts)
    hull_counts.append(len(pts))

hull_counts = np.array(hull_counts)
well_conditioned = hull_counts >= np.percentile(hull_counts, 50)
r_global = np.median(circles[well_conditioned, 2])
print(f"global fixed radius: {r_global:.1f}px (from {well_conditioned.sum()} well-conditioned frames)")

# Refit every frame's center with that radius held fixed
refit = np.array([
    fit_center_fixed_radius(pts, r_global, cx0, cy0)
    for pts, (cx0, cy0, _) in zip(hull_points_list, circles)
])

# Deep partial/near-total frames can still jump -- smooth against a local
# median and snap back any point that strays too far from its neighbors.
smooth_cx = rolling_median(refit[:, 0])
smooth_cy = rolling_median(refit[:, 1])
jump = np.hypot(refit[:, 0] - smooth_cx, refit[:, 1] - smooth_cy)
outliers = jump > 8
final_cx = np.where(outliers, smooth_cx, refit[:, 0])
final_cy = np.where(outliers, smooth_cy, refit[:, 1])
print(f"corrected {int(outliers.sum())} outlier frame(s) via temporal smoothing")

target_cx, target_cy = np.median(final_cx), np.median(final_cy)
crop_r = int(r_global * 1.3)

aligned = []
for img, cx, cy in zip(frames, final_cx, final_cy):
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

print(f"re-aligned {len(aligned)} frames, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
def hull_angular_span_deg(points, cx, cy):
    """How much of the circle's circumference (in degrees) the hull points
    actually cover, found as 360 minus the single biggest angular gap.
    Small span = thin crescent = an unreliable single-frame fit."""
    angles = np.sort(np.arctan2(points[:, 1] - cy, points[:, 0] - cx))
    gaps = np.diff(angles)
    wrap_gap = angles[0] + 2 * np.pi - angles[-1]
    largest_gap = max(gaps.max() if len(gaps) else 0, wrap_gap)
    return np.degrees(2 * np.pi - largest_gap)

spans = np.array([
    hull_angular_span_deg(pts, cx, cy)
    for pts, (cx, cy) in zip(hull_points_list, refit)
])

# Require both a wide arc AND enough points sampling it -- a nominally wide
# but sparsely/noisily sampled arc (few hull vertices) is just as unreliable
# as a short one.
MIN_SPAN_DEG = 120
MIN_HULL_PTS = 20
confident = (spans >= MIN_SPAN_DEG) & (hull_counts >= MIN_HULL_PTS)
idx = np.arange(len(frames))
print(f"{confident.sum()}/{len(frames)} frames trusted directly (span>={MIN_SPAN_DEG}deg, pts>={MIN_HULL_PTS})")
print(f"interpolating the other {(~confident).sum()} frame(s) from their confident neighbors")

# For the low-confidence (near-totality) frames, don't trust the geometric
# fit at all -- interpolate center position in time from confident frames.
final_cx = np.interp(idx, idx[confident], refit[confident, 0])
final_cy = np.interp(idx, idx[confident], refit[confident, 1])

target_cx, target_cy = np.median(final_cx), np.median(final_cy)
crop_r = int(r_global * 1.3)

aligned = []
for img, cx, cy in zip(frames, final_cx, final_cy):
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

print(f"re-aligned {len(aligned)} frames, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
# Slides 56-73 (1-indexed): telescope was covered, nothing real to show -- force pure black.
start_1indexed, end_1indexed = 56, 73
start_idx, end_idx = start_1indexed - 1, end_1indexed - 1
for i in range(start_idx, end_idx + 1):
    aligned[i] = np.zeros_like(aligned[i])
print(f"forced slides {start_1indexed}-{end_1indexed} (indices {start_idx}-{end_idx}) to black")

In [ ]:
exposure_log = []
for path in raw_files:
    with rawpy.imread(path) as raw:
        o = raw.other
    exposure_log.append({
        "file": path.split("/")[-1],
        "iso": o.iso_speed,
        "shutter": o.shutter_speed,
        "time": o.timestamp,
    })

print(f"{'#':>3} {'file':14} {'ISO':>6} {'shutter':>10}   time")
prev_iso, prev_shutter = None, None
for i, e in enumerate(exposure_log):
    changed = (e["iso"] != prev_iso) or (e["shutter"] != prev_shutter)
    marker = " <-- changed" if changed and i > 0 else ""
    print(f"{i + 1:3d} {e['file']:14} {e['iso']:6.0f} {e['shutter']:10.5f}   {e['time']}{marker}")
    prev_iso, prev_shutter = e["iso"], e["shutter"]

In [ ]:
import numpy as np
import cv2

def decode_asshot(path):
    """No auto-brightness (that caused artifacts) and no metadata-based
    exposure guess either -- just the camera's own chosen exposure, demosaiced
    linearly and given a single fixed display gamma ourselves."""
    with rawpy.imread(path) as raw:
        linear = raw.postprocess(
            half_size=True,
            no_auto_bright=True,
            use_camera_wb=True,
            output_bps=16,
            gamma=(1, 1),
        ).astype(np.float64)
    srgb = np.clip(linear / 65535.0, 0, 1) ** (1 / 2.2)
    return (srgb * 255).astype(np.uint8)

frames = []
for i, path in enumerate(raw_files):
    frames.append(decode_asshot(path))
    print(f"\r{i + 1}/{len(raw_files)}", end="")
print(f"\ndecoded {len(frames)} frames (no auto-bright, no forced exposure match yet)")

# --- geometry: locate the sun in each frame ---

def sun_mask(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask

def get_hull_points(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    return cv2.convexHull(largest).reshape(-1, 2).astype(np.float64)

def fit_circle(points):
    x, y = points[:, 0], points[:, 1]
    A = np.column_stack([2 * x, 2 * y, np.ones_like(x)])
    b = x**2 + y**2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy, c = sol
    return cx, cy, np.sqrt(c + cx**2 + cy**2)

def fit_center_fixed_radius(points, r, cx0, cy0, iters=15):
    cx, cy = cx0, cy0
    for _ in range(iters):
        dx, dy = points[:, 0] - cx, points[:, 1] - cy
        d = np.maximum(np.sqrt(dx**2 + dy**2), 1e-6)
        e = d - r
        J = np.column_stack([-dx / d, -dy / d])
        try:
            delta = np.linalg.solve(J.T @ J, -(J.T @ e))
        except np.linalg.LinAlgError:
            break
        cx, cy = cx + delta[0], cy + delta[1]
        if np.linalg.norm(delta) < 1e-4:
            break
    return cx, cy

def hull_angular_span_deg(points, cx, cy):
    angles = np.sort(np.arctan2(points[:, 1] - cy, points[:, 0] - cx))
    gaps = np.diff(angles)
    wrap_gap = angles[0] + 2 * np.pi - angles[-1]
    return np.degrees(2 * np.pi - max(gaps.max() if len(gaps) else 0, wrap_gap))

masks, hull_points_list, circles = [], [], []
for img in frames:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = sun_mask(gray)
    pts = get_hull_points(mask)
    masks.append(mask)
    hull_points_list.append(pts)
    circles.append(fit_circle(pts))
circles = np.array(circles)
hull_counts = np.array([len(p) for p in hull_points_list])

well_conditioned = hull_counts >= np.percentile(hull_counts, 50)
r_global = np.median(circles[well_conditioned, 2])

refit = np.array([
    fit_center_fixed_radius(pts, r_global, cx0, cy0)
    for pts, (cx0, cy0, _) in zip(hull_points_list, circles)
])
spans = np.array([hull_angular_span_deg(pts, cx, cy) for pts, (cx, cy) in zip(hull_points_list, refit)])

MIN_SPAN_DEG, MIN_HULL_PTS = 120, 20
confident = (spans >= MIN_SPAN_DEG) & (hull_counts >= MIN_HULL_PTS)
idx = np.arange(len(frames))
final_cx = np.interp(idx, idx[confident], refit[confident, 0])
final_cy = np.interp(idx, idx[confident], refit[confident, 1])
print(f"{confident.sum()}/{len(frames)} frames trusted directly, {(~confident).sum()} interpolated")

# --- brightness: the sun's actual surface brightness doesn't change as the
# moon covers it (only the exposed *area* shrinks) -- so the mean brightness
# of the illuminated pixels is a direct, measured stand-in for "how this
# frame's exposure compares," no metadata guessing needed. ---
sun_brightness = np.array([
    cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[mask > 0].mean()
    for img, mask in zip(frames, masks)
])
target_brightness = np.median(sun_brightness[confident])
raw_scale = target_brightness / np.maximum(sun_brightness, 1.0)
brightness_scale = np.interp(idx, idx[confident], raw_scale[confident])
print(f"target sun brightness: {target_brightness:.1f}/255, scale range: {brightness_scale.min():.2f}x-{brightness_scale.max():.2f}x")

target_cx, target_cy = np.median(final_cx), np.median(final_cy)
crop_r = int(r_global * 1.3)

aligned = []
for img, cx, cy, scale in zip(frames, final_cx, final_cy, brightness_scale):
    corrected = np.clip(img.astype(np.float64) * scale, 0, 255).astype(np.uint8)
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(corrected, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

# Slides 56-73 (1-indexed): telescope covered, force pure black.
for i in range(55, 73):
    aligned[i] = np.zeros_like(aligned[i])

print(f"re-aligned {len(aligned)} frames, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
TARGET_DISPLAY = 240  # near-white, small headroom left so the disc doesn't fully flatten out
boost = TARGET_DISPLAY / target_brightness
brightness_scale_boosted = brightness_scale * boost
print(f"boosting by {boost:.2f}x so the sun reads ~{TARGET_DISPLAY}/255 (was {target_brightness:.1f}/255)")

aligned = []
for img, cx, cy, scale in zip(frames, final_cx, final_cy, brightness_scale_boosted):
    corrected = np.clip(img.astype(np.float64) * scale, 0, 255).astype(np.uint8)
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(corrected, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

# Slides 56-73 (1-indexed): telescope covered, force pure black.
for i in range(55, 73):
    aligned[i] = np.zeros_like(aligned[i])

print(f"re-aligned {len(aligned)} frames with boosted brightness, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
import base64
from io import BytesIO
from PIL import Image
from IPython.display import HTML

aligned_thumbs = []
for i, img in enumerate(aligned):
    im = Image.fromarray(img)
    buf = BytesIO()
    im.save(buf, format="JPEG", quality=80)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    aligned_thumbs.append(f"data:image/jpeg;base64,{b64}")

frames_js = ",".join(f'"{t}"' for t in aligned_thumbs)
names_js = ",".join(f'"{path.split("/")[-1]}"' for path in raw_files)

html = f"""
<div>
  <img id="aligned-slideshow-img" src="{aligned_thumbs[0]}" style="max-width:480px; display:block;">
  <input id="aligned-slideshow-range" type="range" min="0" max="{len(aligned_thumbs) - 1}" value="0" style="width:480px;">
  <div id="aligned-slideshow-label"></div>
</div>
<script>
(function() {{
  const frames = [{frames_js}];
  const names = [{names_js}];
  const img = document.getElementById("aligned-slideshow-img");
  const range = document.getElementById("aligned-slideshow-range");
  const label = document.getElementById("aligned-slideshow-label");
  function update(i) {{
    img.src = frames[i];
    label.textContent = (parseInt(i) + 1) + " / " + frames.length + " — " + names[i];
  }}
  range.addEventListener("input", (e) => update(e.target.value));
  update(0);
}})();
</script>
"""
HTML(html)

In [ ]:
order = np.argsort(spans)
for i in order[:15]:
    print(f"frame {i:3d} ({raw_files[i].split('/')[-1]}): span={spans[i]:6.1f}deg  hull_pts={len(hull_points_list[i])}")

In [ ]:
for min_span, min_pts in [(60, 0), (90, 0), (90, 15), (120, 20), (150, 25)]:
    c = (spans >= min_span) & (hull_counts >= min_pts)
    print(f"span>={min_span:3d} & pts>={min_pts:2d}: {c.sum():3d}/{len(frames)} confident")